In [0]:
DROP TABLE sandbox.z_jungryo_lee.webos23_24_home_exit_app;
CREATE OR REPLACE TABLE sandbox.z_jungryo_lee.webos23_24_home_exit_app AS (

WITH home_logs AS (
  SELECT
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      CAST(normal_log:visible AS BOOLEAN) AS visible,
      normal_log:app_id AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos24
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
        X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(G4|C4|B4).*'
        OR X_Device_Sales_Model LIKE '%QNED80%'
    )
    AND context_name = 'LSM'
    AND normal_log:app_id = 'com.webos.app.home'

  UNION ALL

  SELECT
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      CAST(normal_log:visible AS BOOLEAN) AS visible,
      normal_log:app_id AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos23
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
        X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(C3|B3).*'
        OR X_Device_Sales_Model LIKE '%UR8300%'
    )
    AND context_name = 'LSM'
    AND normal_log:app_id = 'com.webos.app.home'
),

home_exit AS (
  SELECT
      log_create_time AS home_false_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME
  FROM home_logs
  WHERE visible = false
),

all_app_logs AS (
  SELECT
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      normal_log:app_id AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos23
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
      X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(C3|B3).*'
      OR X_Device_Sales_Model LIKE '%UR8300%'
    )
    AND context_name = 'LSM'
    AND message_id = 'NL_VSC'
    AND normal_log:app_id IS NOT NULL
    AND normal_log:app_id != 'com.webos.app.splash'
    AND normal_log:app_id != 'com.webos.app.volume'
    AND normal_log:app_id != 'com.webos.app.quickinputpicker'
    AND normal_log:app_id != 'com.webos.app.toast'

  UNION ALL

  SELECT
      CAST(log_create_time AS TIMESTAMP) AS log_create_time,
      mac_addr,
      X_User_Number,
      X_Device_Country,
      X_Device_Sales_Model,
      DEVICE_NAME,
      normal_log:app_id AS app_id
  FROM kic_data_ods.tlamp.normal_log_webos24
  WHERE 1=1
    AND date_ym = '2025-05'
    AND log_create_time >= '2025-05-01'
    AND X_Device_Country = 'KR'
    AND (
      X_Device_Sales_Model RLIKE '^OLED[0-9]{2}(G4|C4|B4).*'
      OR X_Device_Sales_Model LIKE '%QNED80%'
    )
    AND context_name = 'LSM'
    AND message_id = 'NL_VSC'
    AND normal_log:app_id IS NOT NULL
    AND normal_log:app_id != 'com.webos.app.splash'
    AND normal_log:app_id != 'com.webos.app.volume'
    AND normal_log:app_id != 'com.webos.app.quickinputpicker'
    AND normal_log:app_id != 'com.webos.app.toast'
),

next_app_ranked AS (
  SELECT
      h.home_false_time,
      h.mac_addr,
      h.X_User_Number,
      h.X_Device_Country,
      h.X_Device_Sales_Model,
      h.DEVICE_NAME,
      a.log_create_time AS next_app_time,
      a.app_id AS next_app_id,
      ROW_NUMBER() OVER (
        PARTITION BY h.mac_addr, h.home_false_time
        ORDER BY a.log_create_time
      ) AS rn
  FROM home_exit h
  LEFT JOIN all_app_logs a
    ON h.mac_addr = a.mac_addr
   AND COALESCE(h.X_User_Number, '') = COALESCE(a.X_User_Number, '')
   AND h.X_Device_Country = a.X_Device_Country
   AND h.X_Device_Sales_Model = a.X_Device_Sales_Model
   AND h.DEVICE_NAME = a.DEVICE_NAME
   AND a.log_create_time > h.home_false_time
   AND a.log_create_time <= h.home_false_time + INTERVAL 10 SECONDS
   AND a.app_id <> 'com.webos.app.home'
),

result AS (
  SELECT
      h.home_false_time,
      DATE(h.home_false_time) AS active_date,
      h.mac_addr,
      h.X_User_Number,
      h.X_Device_Country,
      h.X_Device_Sales_Model,
      h.DEVICE_NAME,
      COALESCE(n.next_app_id, 'NO_APP_WITHIN_10S') AS next_app_id
  FROM home_exit h
  LEFT JOIN next_app_ranked n
    ON h.mac_addr = n.mac_addr
   AND h.home_false_time = n.home_false_time
   AND n.rn = 1
),

device_base AS (
  SELECT
      DEVICE_NAME,
      COUNT(DISTINCT mac_addr) AS total_device_cnt
  FROM result
  GROUP BY DEVICE_NAME
),

session_base AS (
  SELECT
      DEVICE_NAME,
      COUNT(*) AS total_session_cnt
  FROM result
  GROUP BY DEVICE_NAME
),

active_day_base AS (
  SELECT
      DEVICE_NAME,
      COUNT(DISTINCT CONCAT(mac_addr, '_', CAST(active_date AS STRING))) AS total_active_device_day_cnt
  FROM result
  GROUP BY DEVICE_NAME
),

app_stats AS (
  SELECT
      r.DEVICE_NAME,
      r.next_app_id,
      COUNT(*) AS session_cnt,
      COUNT(DISTINCT r.mac_addr) AS device_cnt,
      COUNT(DISTINCT CONCAT(r.mac_addr, '_', CAST(r.active_date AS STRING))) AS active_device_day_cnt,
      d.total_device_cnt,
      s.total_session_cnt,
      a.total_active_device_day_cnt,
      ROUND(COUNT(DISTINCT r.mac_addr) * 1.0 / d.total_device_cnt, 4) AS device_ratio,
      ROUND(COUNT(*) * 1.0 / s.total_session_cnt, 4) AS session_ratio,
      ROUND(COUNT(DISTINCT CONCAT(r.mac_addr, '_', CAST(r.active_date AS STRING))) * 1.0 / a.total_active_device_day_cnt, 4) AS active_day_ratio
  FROM result r
  JOIN device_base d
    ON r.DEVICE_NAME = d.DEVICE_NAME
  JOIN session_base s
    ON r.DEVICE_NAME = s.DEVICE_NAME
  JOIN active_day_base a
    ON r.DEVICE_NAME = a.DEVICE_NAME
  GROUP BY
      r.DEVICE_NAME,
      r.next_app_id,
      d.total_device_cnt,
      s.total_session_cnt,
      a.total_active_device_day_cnt
),

ranked AS (
  SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY DEVICE_NAME
        ORDER BY device_ratio DESC, session_ratio DESC, device_cnt DESC, session_cnt DESC, next_app_id
      ) AS rn
  FROM app_stats
)

SELECT
    DEVICE_NAME,
    next_app_id,
    total_device_cnt,
    total_session_cnt,
    device_cnt,
    session_cnt,
    device_ratio,
    session_ratio,
    active_day_ratio
FROM ranked
WHERE rn <= 20
ORDER BY DEVICE_NAME, rn
);

SELECT *
FROM sandbox.z_jungryo_lee.webos23_24_home_exit_app;
